# 02. Huấn luyện & Ablation Study cho BiLSTM-CRF (Kaggle / Colab)

Notebook của **Tín**: thiết lập môi trường trên Kaggle/Colab, tải dữ liệu `PhoNER_COVID19`, chạy mô hình `BiLSTM-CRF` đầy đủ (`CharCNN + BiLSTM + LinearChainCRF`) và các nhánh đối chứng (Ablation Study).

In [1]:
import os
from pathlib import Path

REPO_URL = "https://github.com/TTins1702/NER-PhoNER_COVID19-UET.git"
BRANCH = "dev"
REPO_DIR = "NER-PhoNER_COVID19-UET"

def _is_repo_root(p: Path) -> bool:
    return (p / "scripts" / "train.py").exists() and (p / "src" / "data" / "labels.py").exists()

if Path.cwd() == Path("/kaggle") and Path("/kaggle/working").exists():
    %cd /kaggle/working
if _is_repo_root(Path("..")):
    %cd ..
if not _is_repo_root(Path(".")):
    if not Path(REPO_DIR).exists():
        !git clone -b {BRANCH} {REPO_URL}
        %cd {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git pull
else:
    !git pull

!pip install -q -r requirements.txt
!python -m src.data.download --output-dir data

# Tự động lấy WANDB_API_KEY từ Colab Secrets / Kaggle Secrets; nếu không có thì chạy offline
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if os.environ.get("WANDB_API_KEY"):
    os.environ.pop("WANDB_MODE", None)
    print("WANDB_API_KEY loaded -> logging online to Weights & Biases")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_API_KEY not set -> running in WANDB_MODE=offline")

Cloning into 'NER-PhoNER_COVID19-UET'...
remote: Enumerating objects: 335, done.
remote: Counting objects: 100% (335/335), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 335 (delta 130), reused 307 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (335/335), 1.13 MiB | 1.16 MiB/s, done.
Resolving deltas: 100% (130/130), done.
/content/NER-PhoNER_COVID19-UET
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
[download] Đang tải: https://github.com/VinAIResearch/PhoNER_COVID19/archive/refs/heads/main.zip
[download] Đã tải 3.60 MB, đang giải nén thư mục data/ ...
[download] Hoàn tất: /content/NER-PhoNER_COVID19-UET/data
[download] Kiểm tra dữ liệu:
  word     train:  5027 câu | 132511 token
  word     dev  :  2000 câu |  56283 token
  word     test :  3000 câu |  85678 token
  syllable train:  5027 câu | 167541 token
  syllable de

In [2]:
# 1. Mô hình đầy đủ: BiLSTM + CharCNN + CRF
!python scripts/train.py --config configs/bilstm_crf.yaml

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttins (SuperMario-UET) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 84kp08tn (0.3s)
wandb: ⣯ setting up run 84kp08tn (0.3s)
wandb: ⣟ setting up run 84kp08tn (0.3s)
wandb: ⡿ setting up run 84kp08tn (0.3s)
wandb: ⢿ setting up run 84kp08tn (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/NER-PhoNER_COVID19-UET/wandb/run-20261120_082815-84kp08tn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm_crf_word
wandb: ⭐️ View project at https://wandb.ai/NER-UET/ner-phoner-covid19-uet
wandb: 🚀 View run at https://wandb.ai/NER-UET/ner-phoner-covid19-uet/runs/84kp08tn
[step 1] epoch=1.0000, train_loss=17.0598, dev_micro

In [3]:
# 2. Ablation 1: Tắt tầng CRF (BiLSTM + CharCNN + Softmax)
!python scripts/train.py --config configs/bilstm_crf.yaml --override experiment_name=bilstm_softmax_word model.use_crf=false training.checkpoint_dir=checkpoints/bilstm_softmax

# 3. Ablation 2: Tắt tầng Character Embedding (BiLSTM + CRF)
!python scripts/train.py --config configs/bilstm_crf.yaml --override experiment_name=bilstm_crf_nochar_word model.use_char=false training.checkpoint_dir=checkpoints/bilstm_crf_nochar

# 4. Ablation 3: Thay CharCNN bằng CharBiLSTM
!python scripts/train.py --config configs/bilstm_crf.yaml --override experiment_name=bilstm_charlstm_crf_word model.char_encoder_type=bilstm training.checkpoint_dir=checkpoints/bilstm_charlstm_crf

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttins (SuperMario-UET) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run tw9lwd8o (0.3s)
wandb: ⣯ setting up run tw9lwd8o (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/NER-PhoNER_COVID19-UET/wandb/run-20261120_083736-tw9lwd8o
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm_softmax_word
wandb: ⭐️ View project at https://wandb.ai/NER-UET/ner-phoner-covid19-uet
wandb: 🚀 View run at https://wandb.ai/NER-UET/ner-phoner-covid19-uet/runs/tw9lwd8o
[step 1] epoch=1.0000, train_loss=0.7317, dev_micro_f1=0.6751, dev_macro_f1=0.4874, dev_bio_violations=468.0000
[step 2] epoch=2.0000, train_loss=0.2875, dev_micro_f1=0

In [4]:
import json

for path in sorted(Path("outputs/metrics").glob("bilstm*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    t = data["test"]
    print(
        f"{data['experiment_name']:28s} | P={t['micro_precision']:.4f} | "
        f"R={t['micro_recall']:.4f} | micro-F1={t['micro_f1']:.4f} | "
        f"macro-F1={t['macro_f1']:.4f} | BIO_viol={t['bio_violations']}"
    )

bilstm_charlstm_crf_word     | P=0.9331 | R=0.9197 | micro-F1=0.9264 | macro-F1=0.8925 | BIO_viol=16
bilstm_crf_nochar_word       | P=0.9256 | R=0.8962 | micro-F1=0.9107 | macro-F1=0.8589 | BIO_viol=24
bilstm_crf_word              | P=0.9271 | R=0.9235 | micro-F1=0.9253 | macro-F1=0.8938 | BIO_viol=19
bilstm_softmax_word          | P=0.9262 | R=0.9087 | micro-F1=0.9174 | macro-F1=0.8814 | BIO_viol=293


In [5]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/NER-UET-outputs/bilstm
!cp -r outputs/metrics /content/drive/MyDrive/NER-UET-outputs/bilstm/
!cp -r checkpoints /content/drive/MyDrive/NER-UET-outputs/bilstm/
!ls -lh /content/drive/MyDrive/NER-UET-outputs/bilstm/metrics

Mounted at /content/drive
total 51K
-rw------- 1 root root  11K Nov 20 09:00 bilstm_charlstm_crf_word.json
-rw------- 1 root root  11K Nov 20 09:00 bilstm_crf_nochar_word.json
-rw------- 1 root root  11K Nov 20 09:00 bilstm_crf_word.json
-rw------- 1 root root  11K Nov 20 09:00 bilstm_softmax_word.json
-rw------- 1 root root  404 Nov 20 09:00 data_entity_counts_word.csv
-rw------- 1 root root  506 Nov 20 09:00 data_entity_length.csv
-rw------- 1 root root  683 Nov 20 09:00 data_oov.csv
-rw------- 1 root root  462 Nov 20 09:00 data_split_stats.csv
-rw------- 1 root root 5.9K Nov 20 09:00 data_stats.json
